# ⚖️ Smart Chargesheet Review & Summarisation Assistant
### हिन्दी कानूनी दस्तावेज़ विश्लेषण सहायक

This notebook demonstrates the **Smart Chargesheet Assistant** – an AI tool that:
1. **Summarises** Hindi police chargesheets into structured summaries
2. **Classifies** the crime type and identifies legal sections
3. **Generates a checklist** of missing/present documents

---

### Architecture
```
┌─────────────────┐     ┌──────────────┐     ┌───────────────────┐
│  Input Document  │────▶│  Chunking    │────▶│  Hierarchical     │
│  (.txt/.pdf/.docx)│    │  (3000 chars) │    │  Summarisation    │
└─────────────────┘     └──────────────┘     │  (LLM per chunk)  │
                                              └───────┬───────────┘
                                                      │
                              ┌───────────────────────┼───────────────────────┐
                              ▼                       ▼                       ▼
                     ┌───────────────┐     ┌───────────────────┐    ┌────────────────┐
                     │  Structured   │     │  Crime Type       │    │  Checklist     │
                     │  Summary      │     │  Classification   │    │  Analysis      │
                     │  (Markdown)   │     │  (LLM + Rules)    │    │  (LLM + Rules) │
                     └───────────────┘     └───────────────────┘    └────────────────┘
```

## 0. Setup & Configuration

In [ ]:
# Install dependencies (run once)
!pip install -q google-genai gradio python-docx PyPDF2

In [ ]:
import os
import sys
import json

# Add project directory to path
PROJECT_DIR = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import config
from processor import (
    extract_text,
    chunk_text,
    load_checklists,
    list_crime_types,
    summarise_chargesheet,
    classify_crime_type,
    detect_crime_type_rules,
    analyse_checklist,
    format_checklist_output,
    format_classification_output,
    process_chargesheet,
)

print(f"LLM Provider: {config.LLM_PROVIDER}")
print(f"Model: {config.GEMINI_MODEL if config.LLM_PROVIDER == 'gemini' else config.OPENAI_MODEL}")

In [ ]:
# ── Set your API Key here ──
# Option 1: Set directly
API_KEY = ""  # <-- Paste your Gemini API key here

# Option 2: From environment variable (preferred)
# API_KEY = os.getenv("GEMINI_API_KEY", "")

if API_KEY:
    config.GEMINI_API_KEY = API_KEY
    os.environ["GEMINI_API_KEY"] = API_KEY
    print("✅ API key configured.")
else:
    print("⚠️ No API key set. Please set API_KEY above or GEMINI_API_KEY environment variable.")
    print("   Get a free key at: https://aistudio.google.com/apikey")

## 1. Load Sample Chargesheet

In [ ]:
# Load the sample robbery chargesheet
sample_file = os.path.join(PROJECT_DIR, "sample_chargesheet_robbery.txt")
chargesheet_text = extract_text(sample_file)

print(f"📄 Document loaded: {os.path.basename(sample_file)}")
print(f"   Characters: {len(chargesheet_text):,}")
print(f"   Est. words:  {len(chargesheet_text.split()):,}")
print(f"   Est. pages:  {max(1, len(chargesheet_text) // 2500)}")
print()
print("── First 500 characters ──")
print(chargesheet_text[:500])

## 2. Document Chunking
For long documents (20-30 pages), we split the text into overlapping chunks for hierarchical summarisation.

In [ ]:
chunks = chunk_text(chargesheet_text)
print(f"Document split into {len(chunks)} chunk(s)")
for i, c in enumerate(chunks):
    print(f"  Chunk {i+1}: {len(c)} characters")

## 3. Crime Type Classification
### 3a. Rule-based Detection (Keyword Matching)

In [ ]:
# Rule-based crime type detection
rule_results = detect_crime_type_rules(chargesheet_text)

print("📊 Rule-based Crime Type Detection Results:")
print("─" * 50)
for r in rule_results:
    bar = "█" * r["score"]
    print(f"  {r['display_name']:45s}  score: {r['score']:3d}  {bar}")

### 3b. LLM-based Classification

In [ ]:
# LLM-based crime type classification
llm_classification = classify_crime_type(chargesheet_text)

print("🔍 LLM Classification Result:")
print(json.dumps(llm_classification, indent=2, ensure_ascii=False))

In [ ]:
# Combined classification output
from IPython.display import Markdown, display
display(Markdown(format_classification_output(llm_classification, rule_results)))

## 4. Document Summarisation

In [ ]:
# Generate structured summary
summary = summarise_chargesheet(chargesheet_text)
display(Markdown(summary))

## 5. Checklist Analysis – Missing Documents
Based on the detected crime type, check which required documents are present and which are missing.

In [ ]:
# Show available crime types and their checklists
crime_types = list_crime_types()
print("Available crime types:")
for ct in crime_types:
    print(f"  • {ct['key']:25s} → {ct['display_name']}")

In [ ]:
# Run checklist analysis for the primary crime type
primary_crime = llm_classification.get("primary_crime_type", rule_results[0]["crime_key"] if rule_results else "theft_robbery")
print(f"Analysing checklist for: {primary_crime}")

checklist_result = analyse_checklist(chargesheet_text, primary_crime)
checklist_md = format_checklist_output(checklist_result, primary_crime)
display(Markdown(checklist_md))

In [ ]:
# Check secondary crime type if applicable
secondary_types = llm_classification.get("secondary_crime_types", [])
for sec_type in secondary_types:
    if sec_type and sec_type != "unknown":
        print(f"\n{'='*60}")
        print(f"Analysing secondary crime type: {sec_type}")
        sec_result = analyse_checklist(chargesheet_text, sec_type)
        sec_md = format_checklist_output(sec_result, sec_type)
        display(Markdown(sec_md))

## 6. Full Pipeline – One-Shot Analysis
Run the complete analysis pipeline in one call.

In [ ]:
# Load the cyber fraud sample
cyber_file = os.path.join(PROJECT_DIR, "sample_chargesheet_cyber.txt")
cyber_text = extract_text(cyber_file)

print(f"📄 Loaded: {os.path.basename(cyber_file)} ({len(cyber_text):,} chars)")

In [ ]:
# Run full pipeline
results = process_chargesheet(cyber_text)

# Display summary
display(Markdown("# 📝 Summary"))
display(Markdown(results["summary"]))

# Display classification
display(Markdown("---"))
display(Markdown(format_classification_output(results["classification"], results["rule_classification"])))

# Display checklists
display(Markdown("---"))
for crime_key, cl in results["checklists"].items():
    display(Markdown(format_checklist_output(cl, crime_key)))

## 7. Bonus: Manual Crime Type Override
Select a crime type manually and regenerate the checklist.

In [ ]:
# ── Change this to test with a different crime type ──
MANUAL_CRIME_TYPE = "assault_hurt"  # Options: theft_robbery, assault_hurt, cyber_fraud, ndps, dowry_harassment, pocso, murder_homicide

manual_result = analyse_checklist(chargesheet_text, MANUAL_CRIME_TYPE)
display(Markdown(format_checklist_output(manual_result, MANUAL_CRIME_TYPE)))

## 8. Launch Gradio UI
Uncomment and run the cell below to launch the interactive web UI.

In [ ]:
# from app import build_app
# app = build_app()
# app.launch(share=False, inline=True)

---
## 🏗️ Future Extensions

| Feature | Description |
|---------|-------------|
| **Hindi → English Translation** | Auto-translate summary to English using LLM |
| **FIR Linking** | Cross-reference with FIR database |
| **Timeline View** | Visual timeline of events from the chargesheet |
| **Risk Scoring** | Score case strength based on present evidence |
| **Multi-document** | Compare multiple chargesheets or link supplementary sheets |
| **OCR Integration** | Direct scan-to-text with Tesseract/EasyOCR for Hindi |
| **Voice Input** | Record oral summary and match with chargesheet |
| **Legal Precedent** | Link to relevant court judgments based on sections |
| **Configurable Rules** | Admin panel to edit checklists per crime type |